## GPT n20 v01 gpt b10 run01 analysis

### requires python >= 3.10

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time
import csv

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [3]:
DATA_FILE = "../../results/n50_examples_large_v01/gpt_v02/gpt_b10_run01.csv"

## Vastustega df

In [4]:
df1 = pd.read_csv(DATA_FILE, encoding="utf-8", sep=",")

In [5]:
len(df1[df1["classification"]=="yes"]) # koht

6336

In [6]:
len(df1[df1["classification"]=="no"]) # mitte koht

3664

In [12]:
# iga prompt jooksutada eelneva prompti "no" peal
(600701 + 345468 + 242119 + 286686 + 213534)/1000000*2.14

3.61340712

In [14]:
600701*5

3003505

In [17]:
# kui timex, alive, event ja org jooksutada kohu loc=no peal
(600701*4)/1000000*2.14

5.1420005600000005

In [19]:
# timex ja alive jooksutada kogu loc=no peal
(412063 + 396263 + 464932)/1000000*2.14

2.7247721200000004

### kas saab olla aeg ja tegija samal ajal

In [22]:
df1[(df1["is_time"]=="yes") & (df1["is_alive"]=="yes")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2


In [23]:
df1[(df1["is_time"]=="yes") & (df1["is_org"]=="yes")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2


### kas abstract koht saab olla aeg

In [7]:
df1[(df1["is_abstract"]=="yes") & (df1["is_time"]=="yes")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
105,18849565,28466497,21,kajastuma,NaN,in,eelmine,eelmistes,"Me hääletasime ainult ühe korra , mis aga praegustes muudatusettepanekutes ei kajastu , sest see kajastus minu mälestust mööda juba eelmistes .",NaN,NaN,NaN,no,"The phrase 'eelmistes' refers to previous proposals or contexts and does not describe a location, hence it is not adverbial of place.",yes,no,no,no,yes,loc
202,6783830,10913630,13,sõitma,NaN,in,augustipimedus,augustipimeduses,"Olen selle kohaga nii kokku kasvanud , et kui ma näiteks siin augustipimeduses rattaga sõidan , tean kõiki teepealseid juurikaid peast .",NaN,NaN,NaN,no,"The phrase 'augustipimeduses' refers to a time period rather than a location, so it is not an adverbial of place.",yes,no,no,no,yes,loc
204,4526783,7276665,5,ristuma,NaN,in,avavoor,avavoorus,"Dortmundi Borussia tee ristub avavoorus laupäeval võõral väljakul Leverkuseni Bayeriga , Müncheni Bayern kohtub reedel ühe autsaideri St.",NaN,event,NaN,no,"The phrase 'avavoorus' refers to an event or time frame, not to a location, hence it is not an adverbial of place.",yes,no,no,no,yes,loc
208,16105875,25002385,11,vabastama,NaN,in,voor,voorus,""" Ashkelon vabastas Babenko pärast tema viletsaid esitusi esimeses neljas voorus , kus ta viskas keskmiselt 3 punkti , "" teatas Iisraeli korvpallianalüütik Shlomi Peri Sõnumilehele .",NaN,NaN,NaN,no,"The phrase 'voorus' references a specific event or stage in a context, not a physical location, so it is not an adverbial of place.",yes,no,no,no,yes,loc
283,8403030,13466997,5,kulgema,NaN,ad,algupool,algupoolel,"Seega kulgeb teise viisirea algupoolel meetrumist lahknev laadirütm , kus paarisarvulisi üksusi võib pidada rõhuliseks .",NaN,time,NaN,no,The phrase 'algupoolel' is not an adverbial of place because it describes a position in a sequence ('early part') rather than a physical location.,yes,no,no,no,yes,loc
474,8908257,14319555,2,algama,NaN,el,Kolmanda,Kolmandast,""" Kolmandast algavad punased .",NaN,NaN,NaN,no,"The phrase 'Kolmandast' is classified as 'no' because it refers to a starting point in a sequence or order, not a physical place.",yes,no,no,no,yes,loc
1253,4841980,7773744,1,hiilgama,NaN,in,jõuluvalgus,Jõuluvalguses,"Jõuluvalguses hiilgavad Mari Adamsoni , Ellen Hanseni , Anu Raua , Aet Ollissaare , Malle Antsoni ja paljude teiste rahvuslikke traditsioone jätkavate kunstnike kudumid .",NaN,NaN,NaN,no,The phrase 'Jõuluvalguses' describes a condition or atmosphere (Christmas light) rather than indicating a specific physical place.,yes,no,no,no,yes,loc
1474,3066873,4922295,1,kindlustama,NaN,in,ideaal,Ideaalis,Ideaalis kindlustab vaba konkurents kõigile kvaliteetse ja piisava toidu odava hinnaga .,NaN,NaN,NaN,no,"The phrase 'Ideaalis' describes an ideal situation or condition, not a physical place, so it is not classified as an adverbial of place.",yes,no,no,no,yes,loc
1947,1022717,1627207,25,kindlustama,NaN,in,ring,ringis,Eesti meeste võrkpallimeistrivõistlustel alistas Audentese Ülikool/Hermann Reisid Tartus sealse Pere Leib Cibuse 3 : 1 ning sisuliselt kindlustas voor enne lõppu kolmandas ringis esikoha .,NaN,NaN,NaN,no,"The phrase 'ringis' refers to a phase or stage in the competition and not a physical location, so it was not classified as an adverbial of place.",yes,no,no,no,yes,loc
2644,5713110,9163874,21,tõstma,NaN,in,lõppfaas,lõppfaasis,""" See sai ka kokku lepitud Euroopa Liidu tippametnikega , et / ... / Eesti selle küsimuse laua peale läbirääkimiste lõppfaasis tõstab , "" märkis ta .",NaN,NaN,NaN,no,"The phrase 'lõppfaasis' describes the timing or phase of an event rather than a location, so it was classified as 'no'.",yes,no,no,no,yes,loc


In [8]:
df1[(df1["is_abstract"]=="yes") & (df1["is_event"]=="yes")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2


In [9]:
at = df1[(df1["is_abstract"]=="yes") & (df1["is_time"]=="yes")]

In [10]:
at.groupby(["verb", "verb_compound", 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,morph_case,count
6,kindlustama,NaN,in,2
17,seisma,NaN,el,2
0,aitama,NaN,in,1
12,nurjuma,NaN,in,1
21,vabastama,NaN,in,1
20,tõstma,NaN,in,1
19,tajuma,NaN,in,1
18,sõitma,NaN,in,1
16,sattuma,NaN,all,1
15,ristuma,NaN,in,1


In [12]:
at[(at["verb"]=="seisma") & (at["morph_case"]=="el")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
7204,669013,1063707,14,seisma,NaN,el,keskaeg,keskajast,"Eesti vanimal maakirikul Ridalas on ajaloos vedanud , sest see on muutumatuna seisnud keskajast peale .",NaN,time,NaN,no,"The phrase 'keskajast' refers to a time period ('since the Middle Ages') and not to a location, so it is not adverbial of place.",yes,no,no,no,yes,loc
7748,2648121,4249231,13,seisma,NaN,el,olevik,olevikust,"Eesti riigipea kõneles valikutest , mille ees seisab Euroopa ning Vana Maailma olevikust ja tulevikust .",NaN,NaN,NaN,no,"The phrase 'olevikust' refers to a temporal concept (the present), not a physical location, so it is not adverbial of place.",yes,no,no,no,yes,loc


### kas saab olla abstract koht ja tegija samal ajal

In [8]:
df1[(df1["is_abstract"]=="yes") & (df1["is_alive"]=="yes")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
65,5552018,8902023,4,alustama,NaN,all,Eesti,Eestile,Läti on sarnaselt Eestile alustanud ettevalmistusi populatsioonigeneetika projektideks vajaliku keskkonna loomiseks .,NaN,location,LOC,no,"The phrase 'Eestile' indicates similarity or comparison, not a location, hence not adverbial of place.",no,yes,no,no,yes,loc
823,8318238,13332325,3,teatama,NaN,all,Aipin,Aipinile,Ma teatasin Aipinile : “ Su põder suri maha .,NaN,NaN,PER,no,"The phrase 'Aipinile' indicates the person being informed and not a location, so it is not an adverbial of place.",no,yes,no,no,yes,loc
824,16250999,25170290,6,eelistama,NaN,all,isik,isikule,"Sina eelistad praegu kindlasti riiki isikule , kes elab oma majas riigi metsamaal , ja väidad , et teised arvavad ka , et tema ei peaks saama .",NaN,NaN,NaN,no,"The phrase 'isikule' indicates the recipient of the preference and not a location, so it is not an adverbial of place.",no,yes,no,no,yes,loc
849,1987162,3169521,6,arutama,NaN,all,isik,isikule,"Valitsus arutab ühes korralduses ühele isikule ja teises korralduses 497 isikule Eesti kodakondsuse andmist , teatas valitsuse pressibüroo .",NaN,NaN,NaN,no,"The phrase 'isikule' refers to a recipient of something rather than a location, so it is not adverbial of place.",no,yes,no,no,yes,loc
1343,5015491,8050472,4,väitma,NaN,el,kohalolnu,kohalolnutest,"Samas väitis üks kohalolnutest , et Jüssi sõitis samuti .",NaN,NaN,NaN,no,"The phrase 'kohalolnutest' is not classified as an adverbial of place because it refers to people present, not to a location.",no,yes,no,no,yes,loc
1345,134650,225354,1,kujunema,NaN,el,Diana,Dianast,"Dianast kujunes popkultuuri ikoon , kõmulehtede meelisobjekt .",NaN,alive,PER,no,The phrase 'Dianast' is not classified as an adverbial of place because it refers to a person and not a spatial or locational context.,no,yes,no,no,yes,loc
1442,3090707,4961831,51,jääma,NaN,el,Döblini,Döblinist,"Selle näitlikustamise eest on Döblinile oma tänuvõlga väljendanud tema jünger Günter Grass , kes on talt tõesti palju õppinud , eeskätt Döblini varajastest vohavatest fantaasiaromaanidest , nagu "" Wang-Luni kolm hüpet "" või "" Mäed , mered ja gigandid "" , mille kohta näiteks Franz Kafka on öelnud : "" Döblinist jääb mulje , nagu mõistaks ta nähtavat maailma millegi täiesti ebatäiuslikuna , mida ta peab alles oma sõnaga loovalt täiendama . """,NaN,NaN,NaN,no,"The phrase 'Döblinist' refers to a person, not a location, so it is not an adverbial of place.",no,yes,no,no,yes,loc
1679,2538379,4072006,5,tulema,NaN,abl,Mõisa,Mõisalt,Peaaegu õige vastus tuleb Mõisalt : 500-600 vahel ehk 550 krooni .,NaN,NaN,PER,no,"The phrase 'Mõisalt' indicates a source or origin of the response rather than a physical location, so it is not classified as adverbial of place.",no,yes,no,no,yes,loc
1702,13434040,21486232,6,loovutama,NaN,all,lugeja,lugejaile,"Oll loovutab valged malendid lahkesti lugejaile , kuid pakub avangustaadiumis aja võitmiseks välja konkreetse seisu , kus sitsiilia kaitse nn . Polugajevski variandist väljunud partiis on tehtud 7 käiku .",NaN,NaN,NaN,no,"The phrase 'lugejaile' refers to individuals (readers) and implies the recipient of an action, not a location, so it is not an adverbial of place.",no,yes,no,no,yes,loc
2075,13032963,20852641,20,osalema,NaN,el,liikmeskond,liikmeskonnast,"Pisut varem peeti sumopühamus Ryogoku Kokugikan , kus nüüd Aki-Basho algab , veel suurem treeningukogunemine osales pool Makuuchi 42mehelisest liikmeskonnast .",NaN,alive,NaN,no,"The phrase 'liikmeskonnast' refers to a group or membership and does not indicate a location or place, so it is not classified as an adverbial of place.",no,yes,no,no,yes,loc


In [9]:
df1[(df1["is_abstract"]=="yes") & (df1["is_org"]=="yes")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
60,11396051,18262044,16,alluma,NaN,all,fraktsioon,fraktsioonile,"Demokraatlikus parlamentarismis jääb iga poliitiline minister alati kahe tule vahele , sest ta allub nii fraktsioonile parlamendis kui ka peaministrile valitsuses .",NaN,NaN,NaN,no,"The phrase 'fraktsioonile' refers to whom the minister is subordinated, not a location, hence not adverbial of place.",no,no,no,yes,yes,loc
433,4535331,7290055,10,saama,NaN,abl,Ford,Fordilt,"Keretarnijateks on ikka Lola ja Reynard , mootoreid saab Fordilt , Hondalt ja Toyotalt .",NaN,NaN,NaN,no,"The phrase 'Fordilt' indicates the source or provider of the engines and not a place, so it is not adverbial of place.",no,no,no,yes,yes,loc
743,2166662,3460881,2,tulema,NaN,all,BREM,BREM-ile,""" BREM-ile ei ole tulnud ühtegi kirja ega kutset konkursile ega ole me ka mujalt selle toimumisest kuulnud , "" ütles Toomas Õispuu .",NaN,NaN,NaN,no,"The phrase 'BREM-ile' indicates the recipient of an action (to whom something was not sent), not a physical location, so it was not classified as an adverbial of place.",no,no,no,yes,yes,loc
748,6552200,10541361,20,tooma,NaN,all,Peugeot,Peugeot'le,"Peugeot' sõitjaist oli Marcus Grönholm sõidu pooleli jätnud juba avapäeva järel , kaheksandaks tulnud Nicolas Bernardi tõi aga Peugeot'le vaid ühe punkti .",NaN,NaN,PER,no,"The phrase 'Peugeot'le' refers to the entity benefiting from an action, not a location, so it was not classified as an adverbial of place.",no,no,no,yes,yes,loc
1072,808276,1289456,5,kuuluma,NaN,all,Finnlinesi,Finnlinesile,"Läänemerel 70 kaubalaevaga opereerivale Finnlinesile kuulub 23 laeva , millest Soome lipu all seilab praegu 14 alust .",NaN,NaN,LOC,no,"The phrase 'Finnlinesile' indicates possession or ownership, not a location, so it is not an adverbial of place.",no,no,no,yes,yes,loc
1237,3617325,5825099,10,takistama,NaN,ad,Microsoft,Microsoftil,"Kohtu otsus aitab kaasa konkurentsi kindlustamisele turul ja takistab Microsoftil saavutada ebaõiglast eelispositsiooni , ütles Reno .",NaN,NaN,LOC,no,"The phrase 'Microsoftil' indicates possession or association, not a physical location, so it is not adverbial of place.",no,no,no,yes,yes,loc
1398,6195759,9949064,22,võtma,NaN,all,Päevaleht,Päevalehele,""" Idee minna Siberisse tuli 14 aastat tagasi aset leidnud ekspeditsioonist , kui eestlaste matmiskohta Norilskis rajati mälestusmemoriaal , "" ütles Päevalehele retkest osa võtnud ja reisikirjanikuna tuntust kogunud Marko Kaldur .",NaN,NaN,ORG,no,"The phrase 'Päevalehele' was classified as 'no' because it designates a recipient of communication, not a spatial or physical place.",no,no,no,yes,yes,loc
1415,14422416,22842624,11,puuduma,NaN,ad,PRIA,PRIAl,"PRIA teabe – ja arendusosakonna juhataja Heli Raametsa sõnul puuduvad PRIAl andmed , kui palju Läänemaal sügisel talivilja maha külvati ja kui palju sellest on hävinud .",NaN,NaN,NaN,no,"The phrase 'PRIAl' indicates possession or a source of data rather than a location, thus it is not adverbial of place.",no,no,no,yes,yes,loc
2107,15376531,24017278,4,käima,NaN,abl,FIFA,FIFA-lt,"Väljakul käis ka FIFA-lt Florasse naasmiseks vajalikku paberit ootav Vjatšeslav Zahovaiko , kuid talle anti Goesi sõnul lihtsalt mängupraktikat .",NaN,NaN,NaN,no,"The phrase 'FIFA-lt' refers to an institution, not to a location, so it was classified as 'no'.",no,no,no,yes,yes,loc
2486,1833625,2918762,1,jätma,NaN,in,Moody,Moody's,Moody's jättis Optivale Forekspanga reitingu,NaN,NaN,LOC,no,The phrase 'Moody's' is not adverbial of place because it refers to a name or entity rather than indicating a place.,no,no,no,yes,yes,loc


In [14]:
ao = df1[(df1["is_abstract"]=="yes") & (df1["is_org"]=="yes")]
aa = df1[(df1["is_abstract"]=="yes") & (df1["is_alive"]=="yes")]

In [13]:
ao.groupby(["verb", "verb_compound", 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,morph_case,count
11,kuuluma,NaN,all,3
18,puuduma,NaN,ad,2
28,tooma,NaN,all,2
17,ostma,NaN,in,2
21,säilima,NaN,ad,2
22,sõnama,NaN,all,2
26,tegema,NaN,all,2
27,tegutsema,NaN,ad,2
29,tulema,NaN,all,1
19,saama,NaN,abl,1


In [15]:
ao[(ao["verb"]=="kuuluma") & (ao["morph_case"]=="all")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
1072,808276,1289456,5,kuuluma,NaN,all,Finnlinesi,Finnlinesile,"Läänemerel 70 kaubalaevaga opereerivale Finnlinesile kuulub 23 laeva , millest Soome lipu all seilab praegu 14 alust .",NaN,NaN,LOC,no,"The phrase 'Finnlinesile' indicates possession or ownership, not a location, so it is not an adverbial of place.",no,no,no,yes,yes,loc
5408,17477316,26791938,6,kuuluma,NaN,all,CIA,CIAle,Ka see lennuk olevat kuulunud CIAle .,NaN,location,NaN,no,"The phrase 'CIAle' indicates a recipient or target rather than a physical place, so it is not adverbial of place.",no,no,no,yes,yes,loc
8049,815650,1301449,8,kuuluma,NaN,all,BMW,BMW-le,Legendaarne Inglise automark Rover kuulub eelmisest aastast BMW-le .,NaN,location,NaN,no,"The phrase 'BMW-le' refers to ownership or belonging, not a physical location, thus it is not adverbial of place.",no,no,no,yes,yes,loc


In [16]:
ao[(ao["verb"]=="puuduma") & (ao["morph_case"]=="ad")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
1415,14422416,22842624,11,puuduma,NaN,ad,PRIA,PRIAl,"PRIA teabe – ja arendusosakonna juhataja Heli Raametsa sõnul puuduvad PRIAl andmed , kui palju Läänemaal sügisel talivilja maha külvati ja kui palju sellest on hävinud .",NaN,NaN,NaN,no,"The phrase 'PRIAl' indicates possession or a source of data rather than a location, thus it is not adverbial of place.",no,no,no,yes,yes,loc
7696,4620236,7425073,17,puuduma,NaN,ad,ER,ER-il,"Kaks aastat tagasi osteti küll Saksamaalt 23 diiselmootorit , kuid seejuures ei arvestatud asjaolu , et ER-il puuduvad vahendid mootorite paigaldamiseks .",NaN,NaN,ORG,no,"The phrase 'ER-il' was classified as 'no' because it refers to an entity or organization rather than a specific location, so it is not an adverbial of place.",no,yes,no,yes,yes,loc


In [17]:
ao[(ao["verb"]=="ostma") & (ao["morph_case"]=="in")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
4257,2280292,3650300,5,ostma,NaN,in,iga,EAS,Väljavalitud piltide kasutusõiguse ostab EAS reklaammaterjalide illustreerimiseks .,NaN,NaN,NaN,no,"'EAS' is not adverbial of place because it refers to an entity (Estonian Enterprise), not a spatial location.",no,no,no,yes,yes,loc
5385,5301162,8504781,21,ostma,NaN,in,Securita,Securitas,"Kuna taastamine pole vanade eksponaatide puhul võimalik , ostis vabaõhumuuseum kindlustuse asemel mullu riigihankekonkursi läbi 3,5 miljoni krooni eest turvafirmalt Securitas elektroonilise valvesüsteemi , mis oli Langi sõnul ka Securitase hooldada .",NaN,NaN,LOC,no,"The phrase 'Securitas' refers to a company name, not a location or a place, so it is not an adverbial of place.",no,no,no,yes,yes,loc


In [18]:
ao[(ao["verb"]=="tulema") & (ao["morph_case"]=="in")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
5413,4406652,7086895,3,tulema,NaN,in,Francai,Francais,"Jah , Francais de Jeux'st tuli meile noor austraallane Mark Renshaw .",NaN,NaN,LOC,no,"The phrase 'Francais' refers to a name and does not indicate a location, so it is not adverbial of place.",no,no,no,yes,yes,loc


In [15]:
aa.groupby(["verb", "verb_compound", 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,morph_case,count
7,loovutama,NaN,all,2
10,saatma,NaN,all,2
1,arutama,NaN,all,2
13,säilima,NaN,ad,2
0,alustama,NaN,all,1
14,sõnama,NaN,all,1
22,vedama,NaN,all,1
21,tõstma,NaN,in,1
20,tulema,NaN,abl,1
19,tooma,NaN,all,1


In [19]:
aa[(aa["verb"]=="loovutama") & (aa["morph_case"]=="all")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
1702,13434040,21486232,6,loovutama,NaN,all,lugeja,lugejaile,"Oll loovutab valged malendid lahkesti lugejaile , kuid pakub avangustaadiumis aja võitmiseks välja konkreetse seisu , kus sitsiilia kaitse nn . Polugajevski variandist väljunud partiis on tehtud 7 käiku .",NaN,NaN,NaN,no,"The phrase 'lugejaile' refers to individuals (readers) and implies the recipient of an action, not a location, so it is not an adverbial of place.",no,yes,no,no,yes,loc
2691,1633394,2600207,15,loovutama,NaN,all,esindaja,esindajale,1993. aasta lõpus Suur-Karja 2 kinnistu tagasi saanud Werner Oldekop loovutas kolmandiku hoonest oma esindajale Selgele .,NaN,alive,NaN,no,"The phrase 'esindajale' refers to a recipient or indirect object and does not describe a location, therefore it is not an adverbial of place.",no,yes,no,no,yes,loc


In [20]:
aa[(aa["verb"]=="saatma") & (aa["morph_case"]=="all")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
2301,7302171,11728706,3,saatma,NaN,all,Saluri,Salurile,Tali saatis Salurile küsimused .,NaN,NaN,PER,no,The phrase 'Salurile' refers to the recipient or indirect object of the action and does not indicate a place.,no,yes,no,no,yes,loc
9298,8345040,13372428,18,saatma,NaN,all,Allik,Allikule,"Kui Allik kunagi Eesti NATOsse astumise vastu sõna võttis , siis tundis Liiv temaga solidaarsust ja saatis Allikule NATO-vastase luuletuse .",NaN,NaN,PER,no,"The phrase 'Allikule' indicates a recipient of the poem (a person), not a physical place, so it is not an adverbial of place.",no,yes,no,no,yes,loc


In [21]:
df1[(df1["classification2"]=="UNK")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation,is_time,is_alive,is_event,is_org,is_abstract,classification2
21,2076950,3314189,11,hoidma,NaN,in,alatoitlus,alatoitluses,"See on olukord , mis hoiab suure hulga eesti lapsi alatoitluses , rääkimata täiskasvanutest .",NaN,NaN,NaN,no,"The phrase 'alatoitluses' describes a condition or state rather than a physical location, so it is not adverbial of place.",no,no,no,no,no,UNK
25,9061330,14569548,9,ennustama,NaN,all,ajakiri,ajakirjale,Keskkooli direktor Tõnu Valdma ennustas seda juba mullu ajakirjale Luup antud intervjuus .,NaN,NaN,NaN,no,"The phrase 'ajakirjale' indicates a recipient of the action rather than a physical location, so it is not adverbial of place.",no,no,no,no,no,UNK
32,6340453,10188183,14,kontrollima,NaN,el,kapitalimaht,kapitalimahust,"Arreteeritud Platon Lebedev juhib Jukose finantskeskuseks olevat gruppi Menatep , mis kontrollib Jukose kapitalimahust 61 protsenti moodustavat eraomanike vara .",NaN,NaN,NaN,no,"The phrase 'kapitalimahust' refers to a proportion of capital, which is not a reference to a location, hence it is not an adverbial of place.",no,no,no,no,no,UNK
34,14437832,22868258,22,müüma,NaN,el,alustamine,alustamisest,"b ) meierei , kes seda võimalust kasutab , ei müü lõssipulbrit sekkumisasutusele nelja nädala jooksul alates käesolevas taandes osutatud tegevuse alustamisest ning teatab inspekteerimisasutusele enne tegevuse alustamist selle alguskuupäeva .",NaN,NaN,NaN,no,"The phrase 'alustamisest' refers to the starting of an action, not a specific place, so it is not an adverbial of place.",no,no,no,no,no,UNK
36,3490887,5621274,15,vähenema,NaN,all,650000,650 000-le,Tööga hõivatud inimeste arv on aga turumajanduslikele suhetele üleminekul vähenenud 200 000 võrra - 850 000-lt 650 000-le .,NaN,NaN,NaN,no,"The phrase '650 000-le' represents a numerical value or quantity, not a location, therefore it is not an adverbial of place.",no,no,no,no,no,UNK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9981,17978857,27357830,5,jääma,NaN,el,100%,100%st,"Kui ELi otsetoetused jäävad 100%st väiksemaks , tuleb toetussumma eraldada Eesti riigieelarvest , et meie põllumeestel oleks liidus võrdsed tingimused .",NaN,NaN,NaN,no,"The phrase '100%st' is not specifying a location, so it's not an adverbial of place.",no,no,no,no,no,UNK
9984,5262886,8442442,6,käima,ära,ad,rekordmadal,rekordmadalal,Telekomi aktsia käis päevasiseselt ära rekordmadalal 70 krooni tasemel .,NaN,NaN,NaN,no,"The phrase 'rekordmadalal' describes a level or state, not a place, so it's not an adverbial of place.",no,no,no,no,no,UNK
9995,981021,1562806,8,levima,NaN,ad,selgitus,selgitusel,"Hansapanga suhtekorraldajate Kristi Liiva ja Ando Noormetsa selgitusel levis tehnilise rikke versioon esmaspäeva hommikul seetõttu , et panga klienditoe töötajatel ei olnud enne turvajuurdluse lõppu juhtunu kohta täpset infot .",NaN,NaN,NaN,no,"The phrase 'selgitusel' explains the means or reason of an action and does not indicate a location, so it is not an adverbial of place.",no,no,no,no,no,UNK
9998,4825664,7748191,4,hüppama,NaN,el,loodetud,loodetust,"Mitmedki mehed hüppasid loodetust kümmekond meetrit vähem , kuni Masahiko Harada tõusis 115meetrise hüppega teiseks .",NaN,NaN,NaN,no,"The phrase 'loodetust' refers to a comparison of expectations, not a specific location, so it is not an adverbial of place.",no,no,no,no,no,UNK


In [22]:
len(df1[df1["is_alive"]=="yes"])

892

In [23]:
len(df1[df1["is_time"]=="yes"])

356

In [24]:
len(df1[df1["is_event"]=="yes"])

227

In [25]:
len(df1[(df1["is_event"]=="yes") & (df1["is_time"]=="no")])

221